# KOLF：PrePR-CT 共表达图与细胞图数据准备

这个 Notebook 将原先 Kang 数据集的流程整理为 KOLF 版本的准备框架。原 Notebook 中的 `condition`、`cell_type`、`SEACell`、`Kang` 路径以及 Linux 服务器路径与 KOLF 数据并不一致，因此已移除这些旧引用。

**重要说明：** 需要先确认输入 `.h5ad` 包含 `gene_target`、`celltype`（或你实际使用的细胞类型字段），并且 `var` 中有项目函数 `Correlation_matrix` / `create_coexpression_graph` 所需的基因注释。此 Notebook 不会擅自推断缺失的基因注释，也不会自动把巨大的表达矩阵转为稠密矩阵。

## 1. 导入依赖并设置路径

In [ ]:
from pathlib import Path
import pickle
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import SEACells

# 修改为当前 PrePR-CT 项目根目录
project_dir = Path(r"E:\\VSCODE\\PrePR-CT\\Cell-Type-Specific-Graphs-main")
sys.path.insert(0, str(project_dir))

import utils
from importlib import reload
reload(utils)
from utils import *

# 输入数据：建议使用已经完成 QC 的 KOLF 数据
data_path = Path(
    r"E:\\1Personal\\New\\科研起步\\large_genetic_perturbation_20260909"
    r"\\02_kolf_ipsc\\KOLF_Pan_Genome_QC_Filtered.h5ad"
)

# 输出文件统一放到项目目录，避免遗留 Linux 服务器路径
graphs_dir = project_dir / "graphs" / "KOLF"
results_dir = project_dir / "Results"
data_dir = project_dir / "Data"
for folder in (graphs_dir, results_dir, data_dir):
    folder.mkdir(parents=True, exist_ok=True)

print("输入数据：", data_path)
print("图输出目录：", graphs_dir)


## 2. 读取数据并统一关键字段

In [ ]:
# 此步骤会读取完整 h5ad；若文件过大导致内存不足，需改用 backed / 分块方案
adata = sc.read_h5ad(data_path)

print("数据规模：", adata.shape)
print("obs 字段：", adata.obs.columns.tolist())
print("var 字段：", adata.var.columns.tolist())

# 将 KOLF 的 gene_target 映射为 PrePR-CT 代码使用的 condition：
# NTC -> control，其他 target -> 对应扰动标签
if "gene_target" not in adata.obs.columns:
    raise KeyError("缺少 obs['gene_target']，请确认输入文件。")
adata.obs["condition"] = adata.obs["gene_target"].astype(str).replace({"NTC": "control"})

# KOLF 常见字段名是 celltype；统一成项目流程中的 cell_type
if "cell_type" not in adata.obs.columns:
    if "celltype" in adata.obs.columns:
        adata.obs["cell_type"] = adata.obs["celltype"].astype(str)
    else:
        raise KeyError("未找到 cell_type 或 celltype 字段，请先确认细胞类型注释。")

# 生成细胞类型 × 条件标签，供后续按扰动分组
adata.obs["cov_drug"] = (
    adata.obs["cell_type"].astype(str) + "_" +
    adata.obs["condition"].astype(str)
)

print("细胞类型分布：")
display(adata.obs["cell_type"].value_counts())
print("对照 / 扰动细胞数：")
display(adata.obs["condition"].value_counts())


## 3. 选取 control 细胞并建立 SEACells

In [ ]:
# SEACells 仅在 control 细胞上拟合，随后将其标签写回完整数据
ctrl_data = adata[adata.obs["condition"] == "control"].copy()

# 使用现有 PCA 表征；如果输入数据没有 X_pca，需要先按项目要求完成预处理
if "X_pca" not in ctrl_data.obsm:
    raise KeyError("ctrl_data.obsm 中没有 X_pca。请先对数据进行 PCA，再运行 SEACells。")

# 目标 metacell 数量按每 50 个 control 细胞约 1 个设置
n_seacells = max(1, int(ctrl_data.n_obs / 50))
model = SEACells.core.SEACells(
    ctrl_data,
    build_kernel_on="X_pca",
    n_SEACells=n_seacells,
    n_waypoint_eigs=10,
    convergence_epsilon=1e-5,
)

model.construct_kernel_matrix()
model.initialize_archetypes()
model.fit(min_iter=10, max_iter=100)

# 保存 control 细胞的 SEACell 标签，并映射回完整 AnnData
adata.obs["SEACell"] = "None"
adata.obs.loc[ctrl_data.obs_names, "SEACell"] = ctrl_data.obs["SEACell"].astype(str).values

print("SEACells 拟合完成；control 中的标签示例：")
display(ctrl_data.obs[["SEACell", "cell_type"]].head())


## 4. 汇总 control metacells

In [ ]:
# 需要 counts layer 来汇总原始计数；若没有该层，先确认 X 是否仍为原始 counts
if "counts" not in ctrl_data.layers:
    raise KeyError("缺少 layers['counts']。请确认输入数据是否保留原始计数层。")

SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(
    ctrl_data,
    model.A_,
    celltype_label="cell_type",
    summarize_layer="counts",
    minimum_weight=0.05,
)

# 统一输出列名，供后续项目函数使用
if "celltype" in SEACell_soft_ad.obs.columns and "cell_type" not in SEACell_soft_ad.obs.columns:
    SEACell_soft_ad.obs.rename(columns={"celltype": "cell_type"}, inplace=True)

seacell_file = data_dir / "KOLF_SEACell_soft.h5ad"
SEACell_soft_ad.write_h5ad(seacell_file)
print("已保存 metacell 数据：", seacell_file)


## 5. 构建并保存细胞类型共表达图

In [ ]:
# 每种细胞类型基于 control metacells 构建一张共表达图
# 相关性绝对值的 99 百分位作为边筛选阈值
cell_type_network = {}

for cell_type in SEACell_soft_ad.obs["cell_type"].astype(str).unique():
    corr_result = Correlation_matrix(
        SEACell_soft_ad,
        cell_type,
        "cell_type",
        hv_genes_cells=None,
        union_HVGs=True,
    )
    corr_matrix = corr_result[0]
    threshold = np.percentile(np.abs(corr_matrix.values), 99.0)

    graph = create_coexpression_graph(
        SEACell_soft_ad,
        corr_result,
        cell_type,
        threshold,
        "gene_name",
    )

    # 图节点的 pos 必须是 torch tensor，后续 create_cells 会用到
    graph.pos = torch.as_tensor(graph.pos, dtype=torch.long)
    graph_file = graphs_dir / f"{cell_type}_coexpr_graph.pkl"
    torch.save(graph, graph_file)
    cell_type_network[cell_type] = graph

    print(f"{cell_type}: nodes={graph.num_nodes}, threshold={threshold:.4g}")

print(f"已保存 {len(cell_type_network)} 张共表达图。")


## 6. 为扰动细胞准备 control 表达并构建训练样本

In [ ]:
# 注意：该步骤会为每个扰动细胞匹配 control 表达。
# 原版代码使用 .X.A 将矩阵稠密化；这里改为按需转换匹配到的切片，
# 但 ctrl_x 层本身仍可能占用大量内存，运行前请评估数据规模。

stim_data = adata[adata.obs["condition"] != "control"].copy()
ctrl_data = adata[adata.obs["condition"] == "control"].copy()

# create_cells 需要图节点的 control 表达均值和方差作为节点特征
for cell_type, graph in cell_type_network.items():
    gene_indices = graph.pos.tolist()
    ctrl_subset = ctrl_data[ctrl_data.obs["cell_type"].astype(str) == str(cell_type), gene_indices]

    X = ctrl_subset.X
    mean_expr = np.asarray(X.mean(axis=0)).ravel()
    if hasattr(X, "multiply"):
        mean_sq = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        mean_sq = np.mean(np.asarray(X) ** 2, axis=0)
    var_expr = np.maximum(mean_sq - mean_expr ** 2, 0)

    graph.x = torch.as_tensor(
        np.column_stack((mean_expr, var_expr)),
        dtype=torch.float32,
    )

# 为每个扰动组合匹配数量相近、SEACell 分布平衡的 control 细胞
# 如果数据规模很大，建议先在小子集测试并确认内存用量
stim_data.layers["ctrl_x"] = np.zeros(stim_data.shape, dtype=np.float32)

for cov_drug in stim_data.obs["cov_drug"].unique():
    cell_type = cov_drug.split("_", 1)[0]
    if cell_type not in cell_type_network:
        continue

    ad_ctrl = ctrl_data[ctrl_data.obs["cell_type"].astype(str) == cell_type]
    ad_stim = stim_data[stim_data.obs["cov_drug"] == cov_drug]
    if ad_ctrl.n_obs == 0 or ad_stim.n_obs == 0:
        continue

    selected = balance_subsample(
        np.asarray(ad_ctrl.obs_names),
        np.asarray(ad_ctrl.obs["SEACell"]),
        ad_stim.n_obs,
    )
    # ctrl_x 只填充当前扰动组对应的细胞行
    stim_data[ad_stim.obs_names, :].layers["ctrl_x"] = ad_ctrl[selected, :].X

# 转为项目训练所需的 PyG Data 对象并保存
cells_train = create_cells(stim_data, cell_type_network, None)
cells_file = data_dir / "cells_KOLF.pkl"
with cells_file.open("wb") as f:
    pickle.dump(cells_train, f)

print(f"已保存 {len(cells_train)} 个细胞图样本：{cells_file}")


## 7. 运行提示与重要限制

- KOLF 数据可能有数百万细胞；`adata = sc.read_h5ad(...)`、`adata.copy()`、`stim_data.layers["ctrl_x"]` 和 `create_cells(...)` 都可能消耗大量内存。
- 当前第 6 节为便于阅读保留了原项目的整体配对思路，但 `ctrl_x` 的完整二维稠密数组对于大型 KOLF 数据通常不可行。建议先用小子集验证流程，再将匹配和写入改为分块/稀疏实现。
- `Correlation_matrix`、`create_coexpression_graph`、`balance_subsample` 和 `create_cells` 的具体输入要求取决于你本地项目 `utils.py` 的实现；本 Notebook 只做语法检查，没有在本地数据上端到端运行。
- KOLF 通常只有一种细胞类型，因此 cell-type-specific graph 可能只有一张图；这不影响单图构建本身，但不会产生多细胞类型间的图差异。